In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import cv2
import shutil
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
HOME_DIR = Path("C:/Users/xiaoh/Desktop/3rd-Paper/Segmentation_Pipeline/outputs/Segformer_qualitative_results")
PREDICTIONS_DIR = Path(os.path.join(HOME_DIR,"Predictions") )
INPUT_IMAGES_DIR = Path(os.path.join(PREDICTIONS_DIR,"big_merged_images"))
OUTPUT_DIR = Path(os.path.join(PREDICTIONS_DIR,"aspect_ratio_analysis"))



# --- Input Data Configuration ---
STEEL_TYPES = {
    "Lower Bainite": INPUT_IMAGES_DIR / "N5325LBITE",
    "Tempered Martensite": INPUT_IMAGES_DIR / "N5440TMITE"
}
# Add any filenames here that you want to exclude from the analysis
FILES_TO_EXCLUDE = [
    'N5325LBITE_0.2-2500X_004.png'
]

# --- Analysis Parameters ---
# Ignore carbides with a pixel area outside this range [MIN, MAX]
CONTOUR_AREA_RANGE = (50, 100000)

# --- Visualization Parameters ---
# Color for the bounding boxes (in BGR format for OpenCV)
BOX_COLOR_BGR = (106, 106, 255)  # A shade of red
FONT_FAMILY = "Times New Roman"
FIGURE_DPI = 300



In [ ]:
# =============================================================================
# 2. CORE FUNCTIONS
# =============================================================================

def analyze_carbide_shapes(image_path: Path) -> tuple[list, list]:
    """
    Analyzes a single image to find carbides and calculate their aspect ratios.

    Args:
        image_path (Path): The path to the input image file.

    Returns:
        A tuple containing:
        - list: A list of aspect ratios for all valid carbides found.
        - list: A list of bounding box coordinates for visualization.
    """
    # Load image in BGR format
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        print(f"Warning: Could not read image file: {image_path}")
        return [], []

    # Convert to grayscale and then create a binary mask using Otsu's thresholding
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, binary_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Find contours of all distinct shapes in the mask
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)

    aspect_ratios = []
    bounding_boxes = []
    min_area, max_area = CONTOUR_AREA_RANGE

    for contour in contours:
        area = cv2.contourArea(contour)
        if not (min_area < area < max_area):
            continue

        # Get the minimum area bounding rectangle for the contour
        rect = cv2.minAreaRect(contour)
        box_points = cv2.boxPoints(rect)
        box = np.int0(box_points)

        # Get width and height, ensuring they are non-zero
        width, height = rect[1]
        if width <= 0 or height <= 0:
            continue
        
        # Calculate aspect ratio as (longer side / shorter side)
        aspect_ratio = max(width, height) / min(width, height)
        aspect_ratios.append(aspect_ratio)
        bounding_boxes.append(box)

    return aspect_ratios, bounding_boxes

def create_highlighted_image(original_image_path: Path, boxes: list) -> np.ndarray:
    """
    Draws bounding boxes on an image for visualization.

    Args:
        original_image_path (Path): Path to the original image.
        boxes (list): A list of bounding box coordinates to draw.

    Returns:
        np.ndarray: A new image (in BGR format) with the highlights.
    """
    img_bgr = cv2.imread(str(original_image_path))
    cv2.drawContours(img_bgr, boxes, -1, BOX_COLOR_BGR, 2)
    return img_bgr

def process_steel_type(steel_name: str, image_folder: Path) -> list:
    """
    Orchestrates the analysis for all images of a specific steel type.

    Args:
        steel_name (str): The display name of the steel (e.g., "Lower Bainite").
        image_folder (Path): The folder containing the images.

    Returns:
        list: A flattened list of all aspect ratios found for this steel type.
    """
    print(f"\nProcessing steel type: {steel_name}...")
    all_ratios = []
    
    # Prepare the output directory for this steel type's highlighted images
    output_image_dir = OUTPUT_DIR / steel_name.replace(" ", "_")
    output_image_dir.mkdir(parents=True, exist_ok=True)

    image_paths = [p for p in image_folder.glob("*.png") if p.name not in FILES_TO_EXCLUDE]

    for image_path in image_paths:
        ratios, boxes = analyze_carbide_shapes(image_path)
        
        if ratios:
            all_ratios.extend(ratios)
            
            # Create and save the highlighted image for visual verification
            highlighted_img = create_highlighted_image(image_path, boxes)
            save_path = output_image_dir / f"{image_path.stem}_highlighted.png"
            cv2.imwrite(str(save_path), highlighted_img)

    print(f"Found {len(all_ratios)} carbides in {len(image_paths)} images.")
    return all_ratios



In [ ]:
# =============================================================================
# 3. MAIN EXECUTION SCRIPT
# =============================================================================

def main():
    """Main function to run the entire analysis and plotting pipeline."""
    
    # --- 1. Setup Environment ---
    # Clean and create the main output directory
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    OUTPUT_DIR.mkdir(parents=True)
    
    # --- 2. Process Data ---
    results_data = {}
    for name, path in STEEL_TYPES.items():
        results_data[name] = process_steel_type(name, path)

    # --- 3. Visualize Results ---
    print("\nGenerating final plot...")
    plt.style.use('default')
    plt.rcParams['font.family'] = FONT_FAMILY

    fig, ax = plt.subplots(figsize=(16, 8))
    
    # Define plot styles for consistency
    plot_styles = {
        "Lower Bainite": {'color': 'red', 'marker': '^', 'facecolors': 'none'},
        "Tempered Martensite": {'color': 'blue', 'marker': '>', 'facecolors': 'none'}
    }

    for name, ratios in results_data.items():
        if not ratios:  # Skip if no carbides were found
            continue
        
        # Calculate statistics
        mean_ratio = np.mean(ratios)
        std_ratio = np.std(ratios)
        
        # Create label text with statistics
        label = f"{name}\n(std: {std_ratio:.2f}, mean: {mean_ratio:.2f})"
        
        # Create the scatter plot
        ax.scatter(range(len(ratios)), ratios, label=label, **plot_styles.get(name, {}))

    # --- 4. Format Plot ---
    ax.set_xlabel('Carbide Index', fontsize=20)
    ax.set_ylabel('Carbide Aspect Ratio', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=15, direction='in', width=2, length=6)
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    
    ax.legend(loc='upper right', fontsize=15, frameon=False)
    
    # Save the final plot
    plot_save_path = OUTPUT_DIR / "Aspect_Ratio_Comparison.png"
    fig.savefig(plot_save_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f"Final plot saved to: {plot_save_path}")

    # --- 5. Save Data to CSV ---
    # Prepare data for DataFrame
    df_data = []
    for name, ratios in results_data.items():
        for ratio in ratios:
            df_data.append({'Steel Type': name, 'Aspect Ratio': ratio})
            
    df = pd.DataFrame(df_data)
    csv_save_path = OUTPUT_DIR / "Aspect_Ratio_Data.csv"
    df.to_csv(csv_save_path, index=False)
    print(f"Raw data saved to: {csv_save_path}")



In [ ]:
if __name__ == "__main__":
    main()